# Stock Markets Analytics Zoomcamp 2026 — Module 1 Homework

Run each code cell, then fill in the **Answer** markdown cell below it based
on what gets printed.

Requirements: `pip install pandas numpy yfinance requests lxml`


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests

pd.set_option("display.max_rows", 100)


## Question 1: S&P 500 Stocks Added to the Index

Which year had the highest number of additions (starting from 2020)?

- 2025
- 2024
- 2023
- 2022

**Additional:** How many current S&P 500 stocks have been in the index for
more than 20 years?


In [4]:
import io

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}
resp = requests.get(url, headers=headers)
tables = pd.read_html(io.StringIO(resp.text))  # <-- the fix is here
df = tables[0]

# Column is usually "Date added"
date_col = [c for c in df.columns if "added" in c.lower()][0]
df["date_added"] = pd.to_datetime(df[date_col], errors="coerce")
df["year_added"] = df["date_added"].dt.year

counts = df[df["year_added"] >= 2020]["year_added"].value_counts().sort_index()
print("Additions per year (>=2020):")
print(counts)

top_year = counts.idxmax()
print(f"\nYear with most additions: {int(top_year)} ({counts.max()} additions)")

cutoff = pd.Timestamp.today() - pd.DateOffset(years=20)
long_tenure = (df["date_added"] < cutoff).sum()
print(f"Stocks in index > 20 years: {long_tenure}")

Additions per year (>=2020):
year_added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

Year with most additions: 2025 (18 additions)
Stocks in index > 20 years: 224


### Answer 1

- Year with highest number of additions (2020+): **`2025`**
- Stocks in index for more than 20 years: **`224`**


## Question 2: Indexes YTD (as of 21 August 2026)

How many indexes (out of 10) have better year-to-date returns than the US
(S&P 500) as of August 21, 2026?

- 1
- 2
- 3
- 4


In [6]:
tickers = {
    "US - S&P 500": "^GSPC",
    "China - Shanghai Composite": "000001.SS",
    "Hong Kong - Hang Seng": "^HSI",
    "Australia - ASX 200": "^AXJO",
    "India - Nifty 50": "^NSEI",
    "Canada - S&P/TSX": "^GSPTSE",
    "Germany - DAX": "^GDAXI",
    "UK - FTSE 100": "^FTSE",
    "Japan - Nikkei 225": "^N225",
    "Mexico - IPC": "^MXX",
    "Brazil - Ibovespa": "^BVSP",
}

ytd = {}
for name, tkr in tickers.items():
    data = yf.download(tkr, start="2026-01-01", end="2026-08-22",
                        progress=False, auto_adjust=False)
    if data.empty:
        print(f"WARNING: no data for {name} ({tkr})")
        continue
    close = data["Close"].squeeze()
    first_close = close.iloc[0]
    last_close = close.iloc[-1]
    ret = (last_close / first_close - 1) * 100
    ytd[name] = float(ret)

s = pd.Series(ytd).sort_values(ascending=False)
print("YTD returns (%) as of 2026-08-21:")
print(s)

us_return = ytd["US - S&P 500"]
better_count = sum(1 for name, r in ytd.items()
                    if name != "US - S&P 500" and r > us_return)
print(f"\nIndexes beating S&P 500 YTD: {better_count} out of 10")

YTD returns (%) as of 2026-08-21:
Japan - Nikkei 225            27.364060
Canada - S&P/TSX              14.856630
US - S&P 500                  11.896237
UK - FTSE 100                  8.697531
Brazil - Ibovespa              6.536106
Germany - DAX                  6.508817
Australia - ASX 200            3.793632
Mexico - IPC                   2.475501
Hong Kong - Hang Seng         -1.249160
China - Shanghai Composite    -2.938152
India - Nifty 50              -7.245892
dtype: float64

Indexes beating S&P 500 YTD: 2 out of 10


### Answer 2

- Number of indexes beating S&P 500 YTD (as of 2026-08-21): **`<fill in>`**


## Question 3: S&P 500 Market Corrections Analysis

Calculate the median drawdown (in %) of significant market corrections in
the S&P 500 index (correction = drop of at least 5% from the most recent
all-time high).

- 8
- 16
- 24
- 32


In [8]:
sp500 = yf.download("^GSPC", start="1950-01-01", progress=False,
                     auto_adjust=False)
close = sp500["Close"].squeeze().dropna()

running_max = close.cummax()
is_ath = close >= running_max  # points that are new all-time highs

ath_dates = close[is_ath].index
ath_prices = close[is_ath]

corrections = []
for i in range(len(ath_dates) - 1):
    start_date = ath_dates[i]
    end_date = ath_dates[i + 1]
    start_price = ath_prices.iloc[i]

    segment = close.loc[start_date:end_date]
    if len(segment) < 2:
        continue
    min_price = segment.min()
    min_date = segment.idxmin()

    drawdown_pct = (start_price - min_price) / start_price * 100
    if drawdown_pct >= 5:
        duration_days = (min_date - start_date).days
        corrections.append({
            "peak_date": start_date,
            "trough_date": min_date,
            "next_ath_date": end_date,
            "drawdown_pct": drawdown_pct,
            "duration_days": duration_days,
        })

corr_df = pd.DataFrame(corrections)
print(f"Number of corrections (>=5% drawdown): {len(corr_df)}")

for pct in [25, 50, 75]:
    dd = np.percentile(corr_df["drawdown_pct"], pct)
    dur = np.percentile(corr_df["duration_days"], pct)
    print(f"{pct}th pct: drawdown={dd:.1f}%, duration={dur:.0f} days")

print("\nTop 10 largest corrections by drawdown:")
print(corr_df.sort_values("drawdown_pct", ascending=False).head(10))

median_drawdown = corr_df["drawdown_pct"].median()
print(f"\nMEDIAN DRAWDOWN: {median_drawdown:.1f}%")

Number of corrections (>=5% drawdown): 74
25th pct: drawdown=6.2%, duration=22 days
50th pct: drawdown=8.0%, duration=40 days
75th pct: drawdown=14.0%, duration=86 days

Top 10 largest corrections by drawdown:
    peak_date trough_date next_ath_date  drawdown_pct  duration_days
56 2007-10-09  2009-03-09    2013-03-28     56.775388            517
54 2000-03-24  2002-10-09    2007-05-30     49.146948            929
24 1973-01-11  1974-10-03    1980-07-17     48.203593            630
22 1968-11-29  1970-05-26    1972-03-06     36.061641            543
65 2020-02-19  2020-03-23    2020-08-18     33.924960             33
35 1987-08-25  1987-12-04    1989-07-26     33.509515            101
15 1961-12-12  1962-06-26    1963-09-03     27.973568            196
27 1980-11-28  1982-08-12    1982-11-03     27.113582            622
68 2022-01-03  2022-10-12    2024-01-19     25.425097            282
18 1966-02-09  1966-10-07    1967-05-04     22.177335            240

MEDIAN DRAWDOWN: 8.0%


### Answer 3

- Median drawdown of corrections (%): **`8`**


## Question 4: Earnings Surprise Analysis for Amazon (AMZN)

Calculate the median 2-day percentage change in stock prices following
positive earnings surprise days.

- 3.35
- 2.35
- 1.35
- 0.35

**Additional:** Is there a correlation between the magnitude of the earnings
surprise and the stock price reaction? Does the market react differently
during bull vs. bear markets?


In [10]:
ticker = "AMZN"
ticker_obj = yf.Ticker(ticker)
earnings = ticker_obj.get_earnings_dates(limit=28)
earnings = earnings.dropna(subset=["Surprise(%)"])  # drop future entry
earnings.index = pd.to_datetime(earnings.index).tz_localize(None).normalize()

prices = yf.download(ticker, start="2020-01-01", progress=False,
                      auto_adjust=False)
close = prices["Close"].squeeze()
close.index = pd.to_datetime(close.index).tz_localize(None).normalize()

trading_days = close.index.sort_values()

results = []
for earn_date, row in earnings.iterrows():
    pos = trading_days.searchsorted(earn_date)
    if pos == 0 or pos >= len(trading_days) - 1:
        continue
    day2_idx = pos + 1 if trading_days[pos] < earn_date else pos

    if day2_idx == 0 or day2_idx >= len(trading_days) - 1:
        continue

    day1 = trading_days[day2_idx - 1]
    day3 = trading_days[day2_idx + 1]

    close_day1 = close.loc[day1]
    close_day3 = close.loc[day3]
    two_day_return = (close_day3 / close_day1 - 1) * 100

    results.append({
        "earnings_date": earn_date,
        "surprise_pct": row["Surprise(%)"],
        "two_day_return_pct": float(two_day_return),
    })

res_df = pd.DataFrame(results)
positive_surprises = res_df[res_df["surprise_pct"] > 0]

median_return = positive_surprises["two_day_return_pct"].median()
print(f"Median 2-day return after POSITIVE surprises: {median_return:.2f}%")

corr = res_df[["surprise_pct", "two_day_return_pct"]].corr()
print("\nCorrelation matrix:")
print(corr)

Median 2-day return after POSITIVE surprises: 1.26%

Correlation matrix:
                    surprise_pct  two_day_return_pct
surprise_pct            1.000000            0.241239
two_day_return_pct      0.241239            1.000000


### Answer 4
Median 2-day return after positive surprises (%): 1.35 (closest match; computed value was 1.26%)
Correlation between surprise magnitude and 2-day return: 0.24 (weak positive)

In [11]:
print(positive_surprises)

   earnings_date  surprise_pct  two_day_return_pct
0     2026-07-30        215.02           19.823514
1     2026-04-29         69.02            2.063914
2     2026-02-05          0.22           -9.730030
3     2025-10-30         25.20            6.044289
4     2025-07-31         27.19           -6.707503
5     2025-05-01         16.77            3.014856
6     2025-02-06         25.29           -2.972437
7     2024-10-31         25.22            2.698074
8     2024-08-01         23.76          -10.204301
9     2024-04-30         17.67           -1.083116
10    2024-02-01         24.38           10.702320
11    2023-10-26         62.22            5.231072
12    2023-08-03         91.19            8.860463
13    2023-04-27         44.29            0.447698
15    2022-10-27         35.53          -10.591388
18    2022-02-03        657.12            4.665611
20    2021-07-29         23.06           -8.338937
21    2021-04-29         67.30            0.257915
22    2021-02-02        100.10 

## Question 5 (Optional): Capstone Project Idea

Describe the capstone project you would like to pursue, considering your
aspirations, ML model predictions, and prior knowledge. Be as specific as
possible (asset class, country, industry vertical, or investment strategy).


### Answer 5

`<fill in your capstone project idea here>`


## Question 6 (Optional): Investigate New Metrics

Using the data sources covered (or others you find relevant), download and
explore a few additional metrics or time series that could be valuable for
your project. Briefly explain why each is useful.


In [ ]:
# Optional: use this cell to explore additional metrics/time series


### Answer 6

`<fill in the metrics you explored and why they're useful>`
